# 🌍 AgentsVille AI Trip Planner

An AI-powered travel planning system that generates and refines vacation itineraries
using Large Language Models (LLMs), structured data validation, and tool-based reasoning.

## How It Works

1. **Collect** traveler preferences (destination, dates, interests, budget)
2. **Gather** simulated weather forecasts and available activities
3. **Generate** an initial day-by-day itinerary with the `ItineraryAgent`
4. **Evaluate** the itinerary across five automated checks
5. **Revise** the itinerary using `ItineraryRevisionAgent` (ReAct loop)
6. **Summarize** the final approved trip

---

## 🔧 Setup

Install dependencies (run once, then restart the kernel):

```bash
pip install -r requirements.txt
```

In [45]:
import importlib
import json
import os
from pathlib import Path

from dotenv import load_dotenv
from openai import OpenAI

import project_lib
load_dotenv()
importlib.reload(project_lib)

from project_lib import (
    VacationInfo,
    TravelPlan,
    get_weather_forecast,
    get_available_activities,
    run_evals,
    ItineraryAgent,
    ItineraryRevisionAgent,
    generate_trip_summary,
    print_itinerary,
    print_eval_results,
)

print('✅ Imports successful')

✅ Imports successful


In [35]:
# ── Configure your Vocareum OpenAI API key ───────────────────────────────────
# Vocareum keys start with "voc-" and must use the Vocareum endpoint.
# Add OPENAI_API_KEY to the workspace .env file or export it in your shell:
#   export OPENAI_API_KEY="voc-..."

if not os.getenv("OPENAI_API_KEY"):
    raise RuntimeError(
        "OPENAI_API_KEY is not set. Add your Vocareum key to .env or export it before running the notebook."
    )

if not os.environ["OPENAI_API_KEY"].startswith("voc-"):
    raise ValueError(
        "This notebook is configured for Vocareum. OPENAI_API_KEY should start with 'voc-'."
    )

client = OpenAI(
    base_url="https://openai.vocareum.com/v1",
    api_key=os.environ["OPENAI_API_KEY"],
)

# Model configuration – use the models supported by the course budget.
MAIN_MODEL = "gpt-4o"
EVAL_MODEL = "gpt-4o-mini"

print(f'✅ Vocareum OpenAI client ready  |  main model: {MAIN_MODEL}  |  eval model: {EVAL_MODEL}')

✅ Vocareum OpenAI client ready  |  main model: gpt-4o  |  eval model: gpt-4o-mini


---
## Step 1 – Define Traveler Preferences

The `VacationInfo` Pydantic model captures everything the planner needs to know
about the traveler: destination, travel dates, interests, budget, and any special
constraints.

In [36]:
vacation_info = VacationInfo(
    destination="AgentsVille",
    start_date="2026-06-10",
    end_date="2026-06-12",
    interests=["culture", "food", "outdoor activities", "entertainment"],
    budget=500.0,
    constraints=["prefer indoor options when raining"],
)

interests_text = ", ".join(vacation_info.interests)
constraints_text = ", ".join(vacation_info.constraints) if vacation_info.constraints else "none"

print("📋 Traveler Preferences")
print(f"   Destination : {vacation_info.destination}")
print(f"   Dates       : {vacation_info.start_date} → {vacation_info.end_date}")
print(f"   Interests   : {interests_text}")
print(f"   Budget      : ${vacation_info.budget:.2f}")
print(f"   Constraints : {constraints_text}")

📋 Traveler Preferences
   Destination : AgentsVille
   Dates       : 2026-06-10 → 2026-06-12
   Interests   : culture, food, outdoor activities, entertainment
   Budget      : $500.00
   Constraints : prefer indoor options when raining


---
## Step 2 – Data Gathering

The system simulates two API calls:

* **Weather forecast** – returns the expected weather condition for each travel date.
* **Available activities** – returns the activities that are available and
  weather-compatible for each date.

In [37]:
# Simulate data retrieval
weather_data = get_weather_forecast(vacation_info)
available_activities = get_available_activities(vacation_info, weather_data)

print('🌤️  Weather Forecast')
for date_str, weather in sorted(weather_data.items()):
    print(f'   {date_str}: {weather}')

print()
print('🎯  Available Activities per Day')
for date_str in sorted(available_activities):
    acts = available_activities[date_str]
    print(f'   {date_str} ({weather_data[date_str]}): {len(acts)} activities available')
    for act in acts[:3]:
        print(f'      • {act["name"]}  –  ${act["cost"]:.2f}')
    if len(acts) > 3:
        print(f'      … and {len(acts) - 3} more')

🌤️  Weather Forecast
   2026-06-10: sunny
   2026-06-11: sunny
   2026-06-12: rainy

🎯  Available Activities per Day
   2026-06-10 (sunny): 18 activities available
      • City Museum Tour  –  $30.00
      • Beach Volleyball  –  $15.00
      • Sunset Boat Cruise  –  $75.00
      … and 15 more
   2026-06-11 (sunny): 18 activities available
      • City Museum Tour  –  $30.00
      • Beach Volleyball  –  $15.00
      • Sunset Boat Cruise  –  $75.00
      … and 15 more
   2026-06-12 (rainy): 11 activities available
      • City Museum Tour  –  $30.00
      • Local Food Tour  –  $45.00
      • Art Gallery Visit  –  $20.00
      … and 8 more


---
## Step 3 – Generate Initial Itinerary

The `ItineraryAgent` sends the traveler preferences, weather forecast, and
available activities to the LLM and asks it to produce a structured
`TravelPlan` (validated by Pydantic).

In [38]:
print('🤖  Calling ItineraryAgent…')

itinerary_agent = ItineraryAgent(client=client, model=MAIN_MODEL)
initial_plan = itinerary_agent.generate(
    vacation_info=vacation_info,
    weather_data=weather_data,
    available_activities=available_activities,
)

print_itinerary(initial_plan)

🤖  Calling ItineraryAgent…

🌍  TRAVEL ITINERARY: AgentsVille

📅  2026-06-10
   • City Museum Tour  –  $30.00
     Explore the rich history and culture of AgentsVille through interactive exhibits and artifacts
   • Local Food Tour  –  $45.00
     Taste the best local cuisine across AgentsVille's vibrant food scene with a knowledgeable guide
   • Sunset Boat Cruise  –  $75.00
     Sail around the bay and watch the breathtaking AgentsVille sunset from the water
   ─ Day total: $150.00

📅  2026-06-11
   • Hiking in National Park  –  $25.00
     Trek through scenic trails in the AgentsVille National Park with stunning panoramic views
   • Wine Tasting Tour  –  $55.00
     Sample award-winning wines from AgentsVille's renowned vineyards and estates
   • Jazz Night at Blue Moon  –  $40.00
     Enjoy live jazz performances and cocktails at AgentsVille's famous Blue Moon venue
   ─ Day total: $120.00

📅  2026-06-12
   • City Museum Tour  –  $30.00
     Explore the rich history and culture of Ag

---
## Step 4 – Evaluate the Itinerary

The evaluation system runs five checks:

| Check | Type | Description |
|---|---|---|
| `budget_accuracy` | Rule-based | Costs tally correctly; total is within budget |
| `city_date_correctness` | Rule-based | Correct destination; all travel dates present |
| `minimum_activities` | Rule-based | At least 2 activities per day |
| `activity_availability` | Rule-based | All activities exist in the catalog for that date |
| `weather_compatibility` | **LLM-based** | Activities suit the day's weather |


In [39]:
print('🔍  Running evaluations on the initial itinerary…')

eval_results = run_evals(
    plan=initial_plan,
    vacation_info=vacation_info,
    weather_data=weather_data,
    available_activities=available_activities,
    client=client,
    model=EVAL_MODEL,
)

print_eval_results(eval_results)

🔍  Running evaluations on the initial itinerary…

📊  EVALUATION RESULTS

✅  PASSED  BUDGET ACCURACY
   Budget check passed: $395.00 is within the $500.00 budget

✅  PASSED  CITY DATE CORRECTNESS
   City and date check passed

✅  PASSED  MINIMUM ACTIVITIES
   Minimum activities check passed: all days have ≥2 activities

✅  PASSED  ACTIVITY AVAILABILITY
   Activity availability check passed

✅  PASSED  WEATHER COMPATIBILITY
   Weather compatibility check passed

OVERALL: ✅  ALL CHECKS PASSED


---
## Step 5 – Revise with the ReAct Agent

The `ItineraryRevisionAgent` follows the ReAct (Reasoning + Acting) framework:

```
THOUGHT → ACTION → OBSERVATION → repeat
```

* **THOUGHT** – the agent reasons about what needs to change
* **ACTION**  – the agent calls a tool (`get_activities_by_date_tool`,
  `calculator_tool`, `run_evals_tool`, or `final_answer_tool`)
* **OBSERVATION** – the agent reads the tool output

The loop continues until the agent calls `final_answer_tool`, signalling that
all checks pass.

In [40]:
revision_agent = ItineraryRevisionAgent(client=client, model=MAIN_MODEL)

final_plan = revision_agent.revise(
    plan=initial_plan,
    vacation_info=vacation_info,
    weather_data=weather_data,
    available_activities=available_activities,
    eval_model=EVAL_MODEL,
)

✅  All checks already pass – no revision needed.


In [46]:
print('\n📋  Revised Itinerary')
print_itinerary(final_plan)

# Run a final evaluation to confirm
print('\n🔍  Final Evaluation')
final_eval = run_evals(
    plan=final_plan,
    vacation_info=vacation_info,
    weather_data=weather_data,
    available_activities=available_activities,
    client=client,
    model=EVAL_MODEL,
)
print_eval_results(final_eval)


📋  Revised Itinerary

🌍  TRAVEL ITINERARY: AgentsVille

📅  2026-06-10
   • City Museum Tour  –  $30.00
     Explore the rich history and culture of AgentsVille through interactive exhibits and artifacts
   • Local Food Tour  –  $45.00
     Taste the best local cuisine across AgentsVille's vibrant food scene with a knowledgeable guide
   • Sunset Boat Cruise  –  $75.00
     Sail around the bay and watch the breathtaking AgentsVille sunset from the water
   ─ Day total: $150.00

📅  2026-06-11
   • Hiking in National Park  –  $25.00
     Trek through scenic trails in the AgentsVille National Park with stunning panoramic views
   • Wine Tasting Tour  –  $55.00
     Sample award-winning wines from AgentsVille's renowned vineyards and estates
   • Jazz Night at Blue Moon  –  $40.00
     Enjoy live jazz performances and cocktails at AgentsVille's famous Blue Moon venue
   ─ Day total: $120.00

📅  2026-06-12
   • City Museum Tour  –  $30.00
     Explore the rich history and culture of AgentsV

---
## Step 6 – Inspect the Agent's Reasoning

The `reasoning_log` records every thought, action, and observation from the
ReAct loop so you can understand how the agent reached its decisions.

In [42]:
print(f'\n🧠  ReAct Reasoning Log ({len(revision_agent.reasoning_log)} entries)\n')
for i, entry in enumerate(revision_agent.reasoning_log, 1):
    entry_type = entry['type'].upper()
    if entry_type == 'THOUGHT':
        print(f'[{i}] 💭 THOUGHT')
        print(f'    {entry["content"][:400].replace(chr(10), " ")}')
    elif entry_type == 'ACTION':
        print(f'[{i}] 🔧 ACTION  →  {entry["tool"]}')
        args_preview = json.dumps(entry.get('args', {}))[:200]
        print(f'    args: {args_preview}')
    elif entry_type == 'OBSERVATION':
        obs_preview = json.dumps(entry['content'])[:200]
        print(f'[{i}] 👁️  OBSERVATION')
        print(f'    {obs_preview}')
    elif entry_type == 'FINAL_ANSWER':
        print(f'[{i}] ✅ FINAL_ANSWER  –  plan submitted')
    print()


🧠  ReAct Reasoning Log (0 entries)



---
## Step 7 – Generate Trip Summary

Once the itinerary passes all checks, the LLM generates a short narrative
summary describing the highlights of the trip.

In [43]:
print('✍️  Generating trip summary…\n')

summary = generate_trip_summary(
    plan=final_plan,
    vacation_info=vacation_info,
    client=client,
    model=MAIN_MODEL,
)

final_plan.summary = summary
print(summary)

✍️  Generating trip summary…

Embark on an unforgettable adventure in AgentsVille from June 10th to June 12th, 2026, all for just $395! Your journey kicks off with a fascinating City Museum Tour, immersing you in the rich local culture before savoring the city’s culinary delights on a Local Food Tour. Cap off your first day with a breathtaking Sunset Boat Cruise. Day two brings the thrill of Hiking in a National Park, followed by a sophisticated Wine Tasting Tour, and an exhilarating evening of Jazz at Blue Moon. As your trip rounds off, enjoy a second dive into the City's Museum, get hands-on in a Cooking Class, and laugh your heart out at an uproarious Comedy Show. Each day is packed with excitement and discovery, promising memories to last a lifetime!


---
## Step 8 – Save the Final Itinerary

Persist the approved plan as a JSON file in the `outputs/` directory.

In [44]:
output_dir = Path('outputs')
output_dir.mkdir(exist_ok=True)

output_path = output_dir / f'itinerary_{vacation_info.destination.lower().replace(" ", "_")}_{vacation_info.start_date}.json'

with open(output_path, 'w', encoding='utf-8') as f:
    f.write(final_plan.model_dump_json(indent=2))

print(f'💾  Itinerary saved to {output_path}')
print()
print('📄  Output preview:')
print(final_plan.model_dump_json(indent=2))

💾  Itinerary saved to outputs/itinerary_agentsville_2026-06-10.json

📄  Output preview:
{
  "destination": "AgentsVille",
  "days": [
    {
      "date": "2026-06-10",
      "activities": [
        {
          "name": "City Museum Tour",
          "cost": 30.0,
          "description": "Explore the rich history and culture of AgentsVille through interactive exhibits and artifacts"
        },
        {
          "name": "Local Food Tour",
          "cost": 45.0,
          "description": "Taste the best local cuisine across AgentsVille's vibrant food scene with a knowledgeable guide"
        },
        {
          "name": "Sunset Boat Cruise",
          "cost": 75.0,
          "description": "Sail around the bay and watch the breathtaking AgentsVille sunset from the water"
        }
      ],
      "day_total_cost": 150.0
    },
    {
      "date": "2026-06-11",
      "activities": [
        {
          "name": "Hiking in National Park",
          "cost": 25.0,
          "description": "T

---
## 🎉 Summary

This notebook demonstrated:

| Feature | Implementation |
|---|---|
| Structured input | `VacationInfo` Pydantic model |
| Simulated data | `get_weather_forecast` / `get_available_activities` |
| AI itinerary generation | `ItineraryAgent` + JSON-mode LLM output |
| Automated evaluation | 5-check eval system (rule-based + LLM) |
| Tool-based reasoning | OpenAI function-calling with 4 tools |
| ReAct loop | `ItineraryRevisionAgent` (THOUGHT→ACTION→OBSERVATION) |
| Structured output | `TravelPlan` Pydantic model |
| Narrative summary | `generate_trip_summary` |

### 🚀 Try It Yourself

Customise the `VacationInfo` in **Step 1** and re-run the notebook to plan
a completely different trip!